# Meeting 12: before the meeting

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 12.

Meeting 11 applied a layer of neurons to data with one line, `X @ W.T + b`, in NumPy, in PyTorch, and on a GPU. This notebook finishes that work and asks what the line costs. You run the layer on a GPU, measure the product on both processors, count its operations, measure how many operations a second several ways of computing reach, find where the numbers of a computation are kept, and predict the time of a layer before running it.

Do it **before meeting 12**, then the second notebook of the meeting, *Rows and columns*, and upload both. The meeting starts from this work: it goes over the main points, and then goes further.

## How this works

- **First, before anything else, make sure that this session has a GPU.** The notebook asks Colab for one: the button **Connect** at the top right says **T4**. Run the first code cell, which says whether the session has a GPU. If it has none, open the **Runtime** menu, choose **Change runtime type**, select **T4 GPU** under *Hardware accelerator*, choose **Save**, and run the first cell again. Colab erases the work of a notebook when it changes its machine, so this comes first. If Colab answers that it cannot connect to a GPU, try again later, or connect without one: every cell of this notebook also runs on a machine without a GPU.
- **Then** choose **Save a copy in Drive** in the **File** menu, so that your changes are kept, and run the cells in order, from the top. To run a cell, click it and press **Shift + Enter**.
- **You may work with your group** of meeting 11, two or three students. Use the number that your group had in meetings 10 and 11; a student without a number chooses one from 1 to 30. **Every student uploads their own copy.**
- **Six parts.** Each has code to run, one thing to write or to change, marked <font color="#c00000"><b>Your turn</b></font>, and a **check** that tests it. If a check reports a problem, its message says what to look at.
- **Six cells ask for your own words**: one marked <font color="#c00000"><b>Your prompt</b></font>, the prompt you give an AI assistant, and five marked <font color="#c00000"><b>Your thinking</b></font>, a few sentences that explain what you measured. **These cells, with five in the second notebook, are what your upload is graded on.** Write them yourself, even when your group worked together.
- **Two posts** on the discussion board of meeting 12, **before the meeting begins**: one after Part 2 and one after Part 6. One member may post for the group, as long as **every post begins with the names of all the members of the group**. The second post is a reply to the first. Under every image, write one sentence that says what the image shows. **A post earns the attention mark of this meeting for everyone it names.** A student who does not post earns it by attending the meeting.
- **Then do the second notebook of the meeting**, [*Rows and columns*](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-12-rows-and-columns.ipynb), and upload **both notebooks** to Canvas together, as the last section says.

<font color="#c00000" size="5"><b>Your turn.</b></font> Double-click this cell and replace this line with your name and the names of the other members of your group.

In [ ]:
# Run this cell first. It says whether this session has a GPU.
import torch

print("PyTorch version:", torch.__version__)
print("Does PyTorch see a GPU?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("The GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Open the Runtime menu, choose Change runtime type, select T4 GPU, choose Save, and run this cell again.")

In [ ]:
# Setup. Run this cell after the GPU check.
import os
import time
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

# The address of the course's files for this unit, and of the folder that holds the images.
SITE = "https://ikoutis.github.io/cs301-labs/unit-2/"
IMAGES = SITE + "images/"

GPU = torch.cuda.is_available()          # True when this session has a GPU


def read_image(file):
    """Read an image file into a NumPy array with values from 0 to 1."""
    return np.array(Image.open(urlopen(IMAGES + file))) / 255.0


def show(pictures, titles):
    """Draw arrays as pictures: an array with two dimensions in grey, one with three in colour."""
    fig, axes = plt.subplots(1, len(pictures), figsize=(5 * len(pictures), 4.2), squeeze=False)
    for ax, picture, title in zip(axes.flat, pictures, titles):
        ax.imshow(np.clip(picture, 0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


def reference_colours(X, K=5, rounds=10):
    """K colours that occur often among the rows of X, from the darkest to the lightest."""
    grey_weights = np.array([0.299, 0.587, 0.114])
    sample = X[::7]                                    # every seventh point is enough
    order = np.argsort(sample @ grey_weights)          # the points, from the darkest to the lightest
    C = np.array([sample[part].mean(axis=0) for part in np.array_split(order, K)])
    for _ in range(rounds):                            # a few rounds of the method k-means
        distance = ((sample[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
        nearest = distance.argmin(axis=1)
        for j in range(K):
            if (nearest == j).any():
                C[j] = sample[nearest == j].mean(axis=0)
    return C[np.argsort(C @ grey_weights)]


def in_words(number):
    """A large number in words: 6.4 million, 1.2 billion, 3.5 trillion."""
    for size, word in ((1e12, "trillion"), (1e9, "billion"), (1e6, "million"), (1e3, "thousand")):
        if number >= size:
            return f"{number / size:.3g} {word}"
    return f"{number:.3g}"


def duration(seconds):
    """A time in the unit that suits it."""
    if seconds < 1:
        return f"{seconds * 1000:.3g} milliseconds"
    if seconds < 120:
        return f"{seconds:.3g} seconds"
    if seconds < 7200:
        return f"{seconds / 60:.3g} minutes"
    if seconds < 172800:
        return f"{seconds / 3600:.3g} hours"
    return f"{seconds / 86400:.3g} days"


print("Setup is done. This session", "has a GPU." if GPU else "has no GPU.")

<font color="#c00000" size="5"><b>Your turn.</b></font> Type your group's number in the next cell, in place of the `0`, and run it: the number that your group had in meetings 10 and 11, so that you work on the same image.

In [ ]:
GROUP = 0        # replace 0 with your group's number, from 1 to 30

assert 1 <= GROUP <= 30, "Type your group's number in place of the 0."

images = pd.read_csv(IMAGES + "images.csv", index_col="number")
picture_info = images.loc[GROUP]
image = read_image(picture_info["file"])
height, width = image.shape[0], image.shape[1]
print(f"Group {GROUP}. Image: {picture_info['title']} ({picture_info['creator']}), {height} rows and {width} columns of pixels.")

## <font color="#1967d2"><b>Part 1: The layer of meeting 11, on a GPU</b></font>

A **neuron** is a rule that converts a point into one number, its **score**: it multiplies each coordinate of the point by a weight, adds the products, and adds a number of its own, the **bias**. A **layer** is a set of neurons that read the same point. With the points as the rows of a matrix `X` (n rows, m columns) and the neurons as the rows of a matrix `W` (K rows, m columns), the scores of all the neurons on all the points are one line, `X @ W.T + b`: n rows and K columns.

The next cell builds the layer of meeting 11. The points are the pixels of your image, three coordinates each (red, green, blue). The five neurons stand for five colours that occur often in the image, and the largest score of a pixel belongs to the colour nearest to it. The cell computes the scores with PyTorch, whose arrays are called **tensors**, on the CPU, and draws every pixel in the colour of its neuron.

In [ ]:
X = image.reshape(-1, 3)                 # one row for each pixel: n points with m = 3 coordinates
n, m = X.shape
K = 5
C = reference_colours(X, K)              # K rows and 3 columns: five colours of the image, darkest first

X_t = torch.from_numpy(X).float()        # the points, as a tensor of float32 numbers
W_t = torch.from_numpy(C).float()        # the weights of the layer: row j is the colour of neuron j
b_t = -0.5 * (W_t ** 2).sum(dim=1)       # the biases: minus half the sum of the squares of each colour
S_t = X_t @ W_t.T + b_t                  # the scores, computed by the CPU: n rows and K columns
print(f"X_t: {tuple(X_t.shape)}   W_t: {tuple(W_t.shape)}   b_t: {tuple(b_t.shape)}   S_t: {tuple(S_t.shape)}   device: {S_t.device}")

winner = S_t.argmax(dim=1).numpy()       # for each pixel, the neuron with the largest score
show([image, C[winner].reshape(height, width, 3)], ["the image", "every pixel in the colour of its neuron"])

A tensor is stored on a **device**: in the memory of the CPU, or in the memory of the GPU, which PyTorch calls `"cuda"`. `X_t.to(device)` **copies** a tensor to a device and returns the copy. An operation needs all its tensors on one device, and the processor of that device carries it out.

<font color="#c00000" size="5"><b>Your turn.</b></font> Complete the function `scores_on` in the next cell. Replace each of the four `...` : the first three lines copy `X_t`, `W_t` and `b_t` to the device, and the last line returns the scores of the layer, computed **with the three copies**. Then run the cell after it, which checks your function. In a session with a GPU, the device is `"cuda"`; without one, it is `"cpu"`, and the same function works.

In [ ]:
device = "cuda" if GPU else "cpu"        # the GPU, when this session has one


def scores_on(device):
    """Copy the points, the weights and the biases to the device, and compute the scores of the layer there."""
    X_d = ...      # Your turn: a copy of X_t on the device
    W_d = ...      # Your turn: a copy of W_t on the device
    b_d = ...      # Your turn: a copy of b_t on the device
    return ...     # Your turn: the scores of the layer, computed with X_d, W_d and b_d

In [ ]:
try:
    S_d = scores_on(device)
except TypeError:
    raise AssertionError("Replace every ... in the function scores_on, and run that cell again.") from None
except RuntimeError as error:
    if "device" in str(error):
        raise AssertionError(
            "Two tensors of one operation are on different devices. Copy all three tensors with .to(device), "
            "and compute the scores with the copies X_d, W_d and b_d.") from None
    raise AssertionError(
        "The shapes do not fit. X_d has shape (n, 3) and W_d has shape (K, 3), and the scores must have n rows "
        "and K columns. Which one has to be transposed?") from None
assert S_d is not ..., "Replace the ... after return with the line that computes the scores."
assert isinstance(S_d, torch.Tensor), "scores_on must return a PyTorch tensor: the scores of the layer."
assert S_d.device.type == device, (
    f"The scores were computed on the device '{S_d.device.type}', not on '{device}'. Compute them with the copies.")
assert tuple(S_d.shape) == (n, K), (
    f"The scores have shape {tuple(S_d.shape)}. They must have shape {(n, K)}: one row for each pixel and one column "
    "for each neuron. Which one has to be transposed?")
assert torch.allclose(S_d.cpu(), S_t, atol=1e-4), (
    "These scores differ from the scores S_t that the CPU computed above. Is the bias added?")
print(f"Correct: the scores were computed on the device '{S_d.device.type}', and .cpu() copies them back. "
      f"The largest difference from the CPU's scores: {float((S_d.cpu() - S_t).abs().max()):.1e}.")

**Why the copies?** Answer in the next cell, in two or three sentences: what does `.to(device)` do to a tensor, and why does `scores_on` copy all three tensors before it computes the scores? What is `.cpu()` needed for in the check above?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 2: The product on both processors</b></font>

The layer of Part 1 is a small product: `W` has 5 rows and 3 columns. The layers of real networks are far larger. The next cell measures `X @ W.T + b` with n points, m coordinates and K neurons all equal to `size`, at four sizes from 512 to 4,096, on the CPU and, if this session has one, on the GPU. The numbers are random. It takes up to half a minute.

Two details are particular to a GPU. PyTorch hands a computation to the GPU and goes on at once, before the result is ready, so `torch.cuda.synchronize()` waits for the GPU before the clock is read. And the first use of a device takes extra time, so one small product is run first and not measured. The cell also measures how long it takes to copy one matrix of size 4,096 into the memory of the GPU.

In [ ]:
def finish(device):
    """Wait until the device has finished its work. A GPU may still be working when PyTorch goes on."""
    if device == "cuda":
        torch.cuda.synchronize()


def time_product(size, device):
    """The average time, in seconds, of X @ W.T + b on the device, with n = m = K = size."""
    X = torch.rand(size, size, device=device)        # random float32 numbers, made on the device
    W = torch.rand(size, size, device=device)
    b = torch.rand(size, device=device)
    repeats = max(1, min(200, 8_000_000_000 // size ** 3))     # many repeats when the product is small
    finish(device)
    start = time.perf_counter()
    for _ in range(repeats):
        S = X @ W.T + b
    finish(device)
    return (time.perf_counter() - start) / repeats


sizes = [512, 1024, 2048, 4096]
devices = ["cpu", "cuda"] if GPU else ["cpu"]
seconds = {}
for d in devices:
    time_product(64, d)                  # one small product that is not measured: the first use of a device is slow
    seconds[d] = [time_product(size, d) for size in sizes]

gpu_name = torch.cuda.get_device_name(0) if GPU else "none"
print(f"X @ W.T + b with n = m = K = size, in seconds. CPU: {os.cpu_count()} logical cores. GPU: {gpu_name}.")
print(f"{'size':>6s}{'multiplications':>18s}{'CPU':>12s}" + (f"{'GPU':>12s}{'CPU / GPU':>12s}" if GPU else ""))
for i, size in enumerate(sizes):
    line = f"{size:6d}{size ** 3:18,d}{seconds['cpu'][i]:12.6f}"
    if GPU:
        line += f"{seconds['cuda'][i]:12.6f}{seconds['cpu'][i] / seconds['cuda'][i]:12.1f}"
    print(line)

if GPU:
    matrix = torch.rand(4096, 4096)                  # one matrix in the memory of the CPU: 4096 x 4096 float32 numbers
    finish("cuda")
    start = time.perf_counter()
    copy = matrix.to("cuda")                         # its copy in the memory of the GPU
    finish("cuda")
    t_copy = time.perf_counter() - start
    megabytes = matrix.element_size() * matrix.nelement() / 1e6
    print(f"\nCopying one matrix of size 4096 ({megabytes:.0f} MB) to the GPU took {t_copy:.4f} seconds; "
          f"the product at that size took the GPU {seconds['cuda'][-1]:.4f} seconds.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(sizes, seconds["cpu"], marker="o", color="#2a78d6", label="CPU")
ax.annotate("CPU", (sizes[-1], seconds["cpu"][-1]), xytext=(8, 0), textcoords="offset points", va="center")
if GPU:
    ax.plot(sizes, seconds["cuda"], marker="s", linestyle="--", color="#eb6834", label="GPU")
    ax.annotate("GPU", (sizes[-1], seconds["cuda"][-1]), xytext=(8, 0), textcoords="offset points", va="center")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(sizes)
ax.set_xticklabels([str(size) for size in sizes])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, position: f"{value:g}"))
ax.margins(x=0.08)
ax.set_xlabel("size: the number of points, of coordinates and of neurons")
ax.set_ylabel("seconds for one product")
ax.set_title("The time of X @ W.T + b. Each step up the axis is 10 times longer.")
ax.grid(True, which="major", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print("Line for the discussion board:")
if GPU:
    print(f"GPU {gpu_name}, CPU with {os.cpu_count()} logical cores. Size 4096: CPU {seconds['cpu'][-1]:.3f} s, "
          f"GPU {seconds['cuda'][-1]:.4f} s, so the GPU is {seconds['cpu'][-1] / seconds['cuda'][-1]:.0f} times as fast. "
          f"Size 512: CPU {seconds['cpu'][0] * 1e3:.1f} milliseconds, GPU {seconds['cuda'][0] * 1e3:.2f} milliseconds. "
          f"Copying one matrix of size 4096 to the GPU: {t_copy:.4f} s.")
else:
    print(f"No GPU in this session. CPU with {os.cpu_count()} logical cores. Size 4096: {seconds['cpu'][-1]:.3f} s. "
          f"Size 512: {seconds['cpu'][0] * 1e3:.1f} milliseconds.")

**How to read the chart.** Both axes are stretched: each size is twice the size before it, and each mark on the vertical axis is ten times the mark below it. The table above the chart holds the same numbers.

<font color="#c00000" size="5"><b>Your turn.</b></font> Read two numbers from the table, and type them in the next cell in place of the `...`:

1. `GPU_FACTOR`: by what factor the GPU is faster than the CPU at size 4,096. If this session has no GPU, type the factor from the table of a member whose session had one, or type `None`.
2. `CPU_GROWTH`: by what factor the time of the CPU grows from size 2,048 to size 4,096. Part 3 explains this factor.

In [ ]:
GPU_FACTOR = ...      # Your turn: the CPU's seconds at size 4,096 divided by the GPU's seconds at size 4,096
CPU_GROWTH = ...      # Your turn: the CPU's seconds at size 4,096 divided by its seconds at size 2,048

In [ ]:
for name, value in (("GPU_FACTOR", GPU_FACTOR), ("CPU_GROWTH", CPU_GROWTH)):
    assert value is not ..., f"Replace the ... after {name} with a number from the table, and run that cell again."
    assert value is None or isinstance(value, (int, float)), f"Type {name} as a number, for example 7.5, not as text."

growth = seconds["cpu"][-1] / seconds["cpu"][-2]
assert CPU_GROWTH is not None and abs(CPU_GROWTH / growth - 1) < 0.1, (
    "CPU_GROWTH does not agree with your table. In the column CPU, divide the seconds in the row of size 4096 "
    "by the seconds in the row of size 2048.")
if GPU:
    factor = seconds["cpu"][-1] / seconds["cuda"][-1]
    assert GPU_FACTOR is not None and abs(GPU_FACTOR / factor - 1) < 0.1, (
        "GPU_FACTOR does not agree with your table. In the row of size 4096, divide the seconds of the CPU by the "
        "seconds of the GPU: the last column, CPU / GPU, holds that factor. It is larger than 1 when the GPU is faster.")
else:
    assert GPU_FACTOR is None or GPU_FACTOR > 0, "Type the factor of a member's table, or None."
print(f"Correct: from size 2,048 to 4,096 the time of the CPU grows by a factor of {growth:.1f}"
      + (f", and at size 4,096 the GPU is {seconds['cpu'][-1] / seconds['cuda'][-1]:.0f} times as fast as the CPU." if GPU else "."))

**Why the GPU is faster, and what a copy costs.** Answer in the next cell, in a few sentences: why is the GPU so much faster than the CPU at size 4,096? Then compare the time of copying one matrix to the GPU, printed under the table, with the time of the product at that size: in what kind of program would the copies cost more than the computation? If your session had no GPU, use the table of a member whose session had one. The page **[CPU and GPU](https://ikoutis.github.io/cs301-labs/unit-2/pages/cpu-and-gpu.html)**, parts 1 and 4, can help.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

<font color="#c00000" size="5"><b>Post now</b></font>. One member posts for the group on the discussion board of meeting 12. Begin the post with the names of all the members of your group: the post earns the attention mark of this meeting for everyone it names. The post contains:

- the line for the discussion board that the cell under the chart printed;
- the chart. To save it, right-click it and choose **Save image as**;
- under the chart, one sentence that says what it shows;
- your two answers, `GPU_FACTOR` and `CPU_GROWTH`, and the name of the GPU, if your session had one.

## <font color="#1967d2"><b>Part 3: Count the operations</b></font>

The next code cell but one computes the scores of a layer with three loops, as in meeting 11. Its innermost line,

`total += W[j, c] * X[i, c]`

carries out one multiplication and one addition. The loops pass through this line once for every point, every neuron and every coordinate: **n × K × m times**. The cell measures the loops for three layers, on random numbers:

| | n, points | m, coordinates | K, neurons |
|---|---|---|---|
| first | 400 | 30 | 20 |
| second | 800 | 30 | 20 |
| third | 800 | 60 | 40 |

<font color="#c00000" size="5"><b>Your turn.</b></font> Before you run the measurement, predict from the count n × K × m, and type your two factors in the next cell:

1. `FACTOR_N_DOUBLED`: from the first layer to the second, by what factor does the number of passes grow?
2. `FACTOR_ALL_DOUBLED`: from the first layer to the third, by what factor does it grow?

In [ ]:
FACTOR_N_DOUBLED = ...       # Your turn: the passes of the second layer divided by the passes of the first
FACTOR_ALL_DOUBLED = ...     # Your turn: the passes of the third layer divided by the passes of the first

In [ ]:
for name, value in (("FACTOR_N_DOUBLED", FACTOR_N_DOUBLED), ("FACTOR_ALL_DOUBLED", FACTOR_ALL_DOUBLED)):
    assert value is not ..., f"Replace the ... after {name} with a number, and run that cell again."
    assert isinstance(value, (int, float)), f"Type {name} as a number, not as text."

layers = [(400, 30, 20), (800, 30, 20), (800, 60, 40)]
passes = [n_ * K_ * m_ for n_, m_, K_ in layers]
assert abs(FACTOR_N_DOUBLED - passes[1] / passes[0]) < 0.01, (
    "From the first layer to the second, only n changes, from 400 to 800; m and K stay. Compute n x K x m for both "
    "layers and divide.")
assert abs(FACTOR_ALL_DOUBLED - passes[2] / passes[0]) < 0.01, (
    "From the first layer to the third, each of the three numbers n, m and K is doubled, and the count is their "
    "product: it is multiplied by 2, then by 2 again, then by 2 again.")
print(f"Correct: {passes[0]:,}, {passes[1]:,} and {passes[2]:,} passes. Now run the next cell and compare the times.")

In [ ]:
def layer_loops(X, W, b):
    """The scores of the K neurons of a layer on the n points of X, one multiplication at a time."""
    n, m = X.shape
    K = W.shape[0]
    S = np.empty((n, K))
    for i in range(n):                   # every point
        for j in range(K):               # every neuron
            total = b[j]
            for c in range(m):           # every coordinate
                total += W[j, c] * X[i, c]
            S[i, j] = total
    return S


layers = [(400, 30, 20), (800, 30, 20), (800, 60, 40)]      # n, m and K of the three layers
rng = np.random.default_rng(0)           # a source of random numbers
loop_times = []
print(f"{'n':>6s}{'m':>5s}{'K':>5s}{'passes, n x K x m':>20s}{'seconds':>10s}{'ratio to the first':>21s}{'nanoseconds per pass':>23s}")
for n_, m_, K_ in layers:
    X_, W_, b_ = rng.random((n_, m_)), rng.random((K_, m_)), rng.random(K_)
    start = time.perf_counter()
    layer_loops(X_, W_, b_)
    loop_times.append(time.perf_counter() - start)
    print(f"{n_:6d}{m_:5d}{K_:5d}{n_ * K_ * m_:20,d}{loop_times[-1]:10.3f}{loop_times[-1] / loop_times[0]:21.1f}"
          f"{loop_times[-1] / (n_ * K_ * m_) * 1e9:23.0f}")

**How to read the result.** The time grows as the number of passes does: by a factor near 2 when n is doubled, and by a factor near 8 when all three numbers are doubled. Measured times vary by some per cent from run to run, so the factors are not exact. The last column, the time of one pass, is nearly the same in the three rows. So

time ≈ (the time of one pass) × n × K × m.

Computer science writes this as **O(nKm)**, read "of the order of n K m". The notation keeps how the number of operations grows with the sizes, and it drops the constant in front, the time of one pass. For a layer with n = m = K = size, the count is size × size × size, the **cube** of the size: when the size doubles, the time is multiplied by 8. That is the factor `CPU_GROWTH` of Part 2.

The cube grows fast, but it is not an *exponential*. An exponential, such as 2<sup>size</sup>, is multiplied by the same factor each time the size grows **by one**; the cube is multiplied by 8 each time the size **doubles**.

The page **[What matrix operations cost](https://ikoutis.github.io/cs301-labs/unit-2/pages/matrix-costs.html)** lists this count for the operations of meeting 11.

**A batch twice as large.** A network applies a layer of K = 512 neurons to a batch of n = 256 images, each with m = 3,072 numbers. Answer in the next cell, in a few sentences, from the count: if the batch is made twice as large and the layer half as large, how does the time of one product change, and why? And if n, m and K are all made twice as large?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 4: What a second is worth</b></font>

The count says how the time **grows**. It does not say how long one operation takes. That is measured as the number of **arithmetic operations per second**, where each multiplication and each addition counts as one operation:

operations per second = (number of operations) ÷ (seconds).

<font color="#c00000" size="5"><b>Your turn.</b></font> Complete the function `product_rate` in the next cell: it receives the size of a product `X @ W.T`, with n = m = K = `size`, and the seconds it took, and it returns the number of operations per second. Count as in Part 3: each pass of the innermost line is one multiplication and one addition.

In [ ]:
def product_rate(size, seconds):
    """Arithmetic operations per second of the product X @ W.T with n = m = K = size, which took `seconds`."""
    return ...     # Your turn: the number of operations of the product, divided by the seconds

In [ ]:
got = product_rate(4096, 0.05)
assert got is not ..., "Replace the ... after return with your formula, and run that cell again."
right = 2 * 4096 ** 3 / 0.05
if got < 1:
    raise AssertionError("Operations per second: divide the number of operations by the seconds, not the seconds by "
                         "the operations.")
if abs(got / (right / 2) - 1) < 1e-6:
    raise AssertionError("This counts one operation for each pass. Each pass of the innermost line is one multiplication "
                         "and one addition: two operations.")
if abs(got / (right / 4096) - 1) < 1e-6 or abs(got / (right / 8192) - 1) < 1e-6:
    raise AssertionError("A product with n = m = K = size has size x size x size passes, not size x size.")
assert abs(got / right - 1) < 1e-6, (
    "The rate is not right. A product with n = m = K = size has size x size x size passes, each one multiplication and "
    "one addition; divide that number of operations by the seconds.")
print(f"Correct: the product at size 4,096 in 0.05 seconds is {in_words(got)} operations per second.")

The next cell compares five ways of computing:

1. **Python, three loops:** the first layer of Part 3;
2. **NumPy, `a + b` on long arrays:** two arrays of about 16 million numbers each are added, four times;
3. **NumPy, `a + b` on short arrays:** two arrays of about 16 thousand numbers each are added 4,096 times, which is the same number of additions as in the line above;
4. **the CPU, the product** `X @ W.T + b` at size 4,096, from your table of Part 2;
5. **the GPU, the same product**, from the same table, if your session has a GPU.

In [ ]:
rng = np.random.default_rng(1)
rates = {}                               # for each way of computing: arithmetic operations per second

# 1. Python, three loops: one multiplication and one addition for each pass
n_, m_, K_ = layers[0]
rates["Python: three loops"] = 2 * n_ * K_ * m_ / loop_times[0]

# 2. NumPy, a + b on long arrays: one addition for each pair of numbers
long = 2 ** 24                           # 16,777,216 numbers in each array
a = rng.random(long, dtype=np.float32)
b = rng.random(long, dtype=np.float32)
c = np.empty(long, dtype=np.float32)
np.add(a, b, out=c)                      # c = a + b, with the result written into the array c; not measured
start = time.perf_counter()
for _ in range(4):
    np.add(a, b, out=c)
rates["NumPy: a + b, long arrays"] = 4 * long / (time.perf_counter() - start)

# 3. NumPy, a + b on short arrays: the same number of additions
short = 2 ** 14                          # 16,384 numbers in each array
a, b, c = a[:short].copy(), b[:short].copy(), c[:short].copy()
calls = 4 * long // short
np.add(a, b, out=c)
start = time.perf_counter()
for _ in range(calls):
    np.add(a, b, out=c)
rates["NumPy: a + b, short arrays"] = calls * short / (time.perf_counter() - start)

# 4 and 5. The product at size 4,096, from Part 2. The bias adds 4,096 x 4,096 additions, too few to count.
rates["CPU: X @ W.T + b"] = product_rate(4096, seconds["cpu"][-1])
if GPU:
    rates["GPU: X @ W.T + b"] = product_rate(4096, seconds["cuda"][-1])

slowest = min(rates.values())
print(f"{'way of computing':30s}{'operations per second':>24s}{'ratio to the slowest':>23s}")
for name, value in rates.items():
    print(f"{name:30s}{in_words(value):>24s}{value / slowest:23,.0f}")

In [ ]:
names = list(rates)
values = list(rates.values())

fig, ax = plt.subplots(figsize=(9, 0.62 * len(names) + 1.5))
ax.hlines(names, min(values) / 3, values, color="#c3c2b7", linewidth=1)      # a thin line from the edge to each mark
ax.plot(values, names, "o", color="#2a78d6", markersize=9)
for name, value in zip(names, values):
    ax.annotate(in_words(value), (value, name), xytext=(10, 0), textcoords="offset points", va="center")
ax.set_xscale("log")
ax.set_xlim(min(values) / 3, max(values) * 40)
ticks = [10.0 ** e for e in range(6, 15) if min(values) / 3 <= 10.0 ** e <= max(values) * 40]
ax.set_xticks(ticks)
ax.set_xticklabels([in_words(tick) for tick in ticks])
ax.invert_yaxis()                        # the first way of computing at the top
ax.set_xlabel("arithmetic operations per second (each mark on the axis is 10 times the one before)")
ax.set_title("How many additions and multiplications a second?")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
fig.tight_layout()
plt.show()

print("Line for the discussion board:")
print("Operations per second. " + "; ".join(f"{name}: {in_words(value)}" for name, value in rates.items()) + ".")

**How to read the chart.** Every mark counts the same thing: additions and multiplications of two numbers in one second. The axis is stretched: each mark on it is ten times the one before. The table above the chart holds the same numbers.

- **Python against NumPy.** The loops run in the interpreter of Python, which does a great deal of work around each operation. NumPy's loops run in code made for the processor.
- **Long arrays against short arrays.** These two rows run the same additions, by the same code. Only the place of the numbers differs. A processor has a small and fast memory of its own, called the **cache**. The short arrays fit into it and stay there. The long arrays do not fit: each number has to be fetched from the main memory of the computer, is used in **one** addition, and the result is sent back. The processor spends most of its time waiting for numbers. Such a computation is **memory-bound**: the memory sets its speed, not the arithmetic.
- **The product.** In `X @ W.T` a number is not used once: each entry of `X` is multiplied with a weight of every neuron. The product is programmed to work on blocks of the two matrices that fit into the cache, and to use each number many times while it is there. The processor spends its time on arithmetic. Such a computation is **compute-bound**.
- **The GPU.** Thousands of small cores work on the same product. The data sheet of the T4, the GPU that Colab lends, gives 8.1 trillion operations a second at its peak. Compare your mark with that number: a real program reaches a part of the peak.

The notation O( ) is a first estimate of the cost of a computation, and the right one to begin with. The marks of the chart show what it leaves out: how the program is run, where the numbers are, and which processor does the work. The page **[CPU and GPU](https://ikoutis.github.io/cs301-labs/unit-2/pages/cpu-and-gpu.html)** shows what is inside the two processors.

## <font color="#1967d2"><b>Part 5: Where the numbers are</b></font>

Part 4 found that the same additions run faster on short arrays than on long ones. This part measures the rate of `a + b` at sixteen lengths of the arrays, from about a thousand numbers to 33 million, with a function that an AI assistant writes for you.

<font color="#c00000" size="5"><b>Your turn.</b></font> Ask an AI assistant (Gemini in Colab, ChatGPT, Claude, or another) to write the function `add_rate`. State all of these requirements in your prompt:

- the function is named `add_rate` and takes two arguments, `length` and `repeats`;
- it makes two NumPy arrays `a` and `b` of `length` random numbers of the type `float32`, and a third array `c` of the same length for the result;
- it computes `np.add(a, b, out=c)` once without measuring it, and then `repeats` times, measured with `time.perf_counter()`;
- it returns the number of additions per second: `length` × `repeats`, divided by the seconds that the `repeats` additions took;
- it has no loop over the numbers of the arrays: its only loop repeats the whole addition.

Paste your prompt into the next cell, and the function into the code cell after it. Then run the check.

<font color="#c00000" size="5"><b>Your prompt.</b></font> Double-click this cell and replace this line with the prompt you gave to the AI assistant. If the first function did not pass the check, add one sentence on what you changed.

In [ ]:
# Your turn: paste the function add_rate here, then run this cell.

In [ ]:
assert "add_rate" in globals(), "Paste the function add_rate into the cell above and run that cell."


def reference_rate(length, repeats):
    """np.add on two arrays of float32 numbers, as the requirements describe it: additions per second."""
    a, b, c = np.ones(length, np.float32), np.ones(length, np.float32), np.empty(length, np.float32)
    np.add(a, b, out=c)
    start = time.perf_counter()
    for _ in range(repeats):
        np.add(a, b, out=c)
    return length * repeats / (time.perf_counter() - start)


got = max(add_rate(2 ** 14, 500) for _ in range(3))           # the best of three, as for the reference
assert isinstance(got, (int, float, np.floating)) and got > 0, "add_rate must return one positive number: additions per second."
expected = max(reference_rate(2 ** 14, 500) for _ in range(3))
if got < 1:
    raise AssertionError("add_rate returns a number smaller than 1: seconds, perhaps. It must return the additions per "
                         "second: length x repeats, divided by the seconds.")
if got < expected / 100:
    raise AssertionError("add_rate gives hundreds of times fewer additions a second than np.add reaches on these arrays. "
                         "Does it count the additions of one call only? There are length x repeats of them. Or does it "
                         "loop over the numbers of the arrays?")
if got < expected / 3:
    raise AssertionError("add_rate gives far fewer additions a second than np.add reaches on these arrays. Does it loop over "
                         "the numbers, or make the arrays inside the measured part?")
if got > expected * 3:
    raise AssertionError("add_rate gives far more additions a second than np.add reaches on these arrays. Count the "
                         "additions as length x repeats, and divide by the seconds of all the repeats together.")
print(f"Correct: add_rate gives {in_words(got)} additions a second for two arrays of 16,384 numbers.")

In [ ]:
import glob

lengths = [2 ** k for k in range(10, 26)]                 # from 1,024 to 33,554,432 numbers in each array
sweep = [add_rate(length, max(1, 2 ** 26 // length)) for length in lengths]      # about 67 million additions each
megabytes = [3 * 4 * length / 1e6 for length in lengths]   # a, b and c: three arrays of numbers of 4 bytes

caches = []                                                 # the caches of this processor, where the system tells them
for folder in sorted(glob.glob("/sys/devices/system/cpu/cpu0/cache/index*")):
    try:
        kind = open(folder + "/type").read().strip()
        level = open(folder + "/level").read().strip()
        size = open(folder + "/size").read().strip()
    except OSError:
        continue
    if kind != "Instruction":
        factor = {"K": 1e3 * 1.024, "M": 1e6 * 1.048576}.get(size[-1], 1e-6)
        caches.append((f"L{level}", float(size.rstrip("KM")) * factor / 1e6))

print(f"{'numbers in each array':>22s}{'MB of the three arrays':>24s}{'additions per second':>22s}")
for length, mb, rate in zip(lengths, megabytes, sweep):
    print(f"{length:22,d}{mb:24.3f}{in_words(rate):>22s}")
print("Caches of this processor:", ", ".join(f"{name} {mb:g} MB" for name, mb in caches) if caches
      else "the system of this machine does not tell them.")

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(megabytes, [rate / 1e9 for rate in sweep], marker="o", color="#2a78d6")
for name, mb in caches:
    ax.axvline(mb, color="#888888", linestyle=":", linewidth=1)
    ax.annotate(f"{name}: {mb:g} MB", (mb, max(sweep) / 1e9), xytext=(4, -4), textcoords="offset points", fontsize=9, va="top")
ax.set_xscale("log")
ax.set_xlabel("MB of the three arrays a, b and c (each mark on the axis is 10 times the one before)")
ax.set_ylabel("billions of additions a second")
ax.set_ylim(0, max(sweep) / 1e9 * 1.15)
ax.set_title("The rate of a + b, against the size of the arrays")
ax.grid(True, which="major", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

print("Line for the discussion board:")
print(f"a + b: {in_words(max(sweep))} additions a second at best, at {megabytes[sweep.index(max(sweep))]:.2g} MB; "
      f"{in_words(sweep[-1])} at {megabytes[-1]:.0f} MB.")

**How to read the chart.** Every point runs the same additions, by the same function; only the size of the arrays changes. A processor keeps the numbers it works on in a hierarchy of memories. The smallest and fastest are the levels of its **cache**: **L1**, tens of kilobytes beside each core; **L2**, about a megabyte; **L3**, tens of megabytes, shared by the cores. Behind them is the **main memory** of the computer, gigabytes, and slower to reach. Where the cell found the sizes of this processor's caches, the chart marks them with dotted lines.

At the far left of the chart, the arrays are so short that another cost shows: each call of `np.add` takes a fixed time of its own, whatever the length of the arrays, and a few thousand additions do not hide it.

**Where the numbers are.** Answer in the next cell, in a few sentences: at which sizes of the arrays does the rate of `a + b` drop, and what does that say about where the numbers are kept? Why does the product `X @ W.T` of Part 4 not slow down in the same way when its matrices are larger than the cache?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 6: Predict the time of a layer, then measure it</b></font>

The count and the rates together predict the time of a computation before it is run:

time ≈ (number of operations) ÷ (operations per second).

A layer of K neurons on n points with m coordinates takes 2 × n × K × m operations. For example, a layer of K = 1,000 neurons applied to n = 60,000 small grey images of 28 by 28 pixels, so that m = 784, takes about 94 billion.

<font color="#c00000" size="5"><b>Your turn.</b></font> Choose a layer of your own: replace the three numbers in the next cell. Keep the product below 1,000 billion operations, so that the CPU finishes it in a reasonable time, and below 2 GB of numbers. Then run the cell after it. It predicts the time of your layer with the three loops of Python, on the CPU and on the GPU, from your rates of Part 4, and then measures it on the CPU and on the GPU. The three loops are not run: they would take too long.

In [ ]:
n_layer, m_layer, K_layer = 60_000, 784, 1_000      # Your turn: the points, their coordinates, and the neurons of your layer

In [ ]:
for name, value in (("n_layer", n_layer), ("m_layer", m_layer), ("K_layer", K_layer)):
    assert isinstance(value, int) and value > 0, f"{name} must be a whole number larger than 0."
operations = 2 * n_layer * K_layer * m_layer
gigabytes = 4 * (n_layer * m_layer + K_layer * m_layer + n_layer * K_layer) / 1e9     # X, W and the scores, in float32
assert operations <= 1e12, f"This layer takes {in_words(operations)} operations: choose smaller numbers, below 1,000 billion."
assert gigabytes <= 2, f"The numbers of this layer take {gigabytes:.1f} GB: choose smaller numbers, below 2 GB."


def time_layer(n, m, K, device):
    """The time, in seconds, of X @ W.T + b on the device, for random X with n rows and m columns and K neurons."""
    X = torch.rand(n, m, device=device)
    W = torch.rand(K, m, device=device)
    b = torch.rand(K, device=device)
    S = X @ W.T + b                      # once, not measured
    finish(device)
    start = time.perf_counter()
    S = X @ W.T + b
    finish(device)
    return time.perf_counter() - start


ways = [("Python: three loops", "Python: three loops", None), ("CPU", "CPU: X @ W.T + b", "cpu")]
if GPU:
    ways.append(("GPU", "GPU: X @ W.T + b", "cuda"))
print(f"A layer of K = {K_layer:,} neurons on n = {n_layer:,} points with m = {m_layer:,} coordinates: "
      f"{in_words(operations)} operations, {gigabytes:.2f} GB of numbers.")
print(f"{'way of computing':22s}{'predicted':>20s}{'measured':>20s}{'measured / predicted':>23s}")
board = []
for label, row, d in ways:
    predicted = operations / rates[row]
    if d is None:
        print(f"{label:22s}{duration(predicted):>20s}{'not run':>20s}")
        board.append(f"{label}: predicted {duration(predicted)}")
        continue
    measured = time_layer(n_layer, m_layer, K_layer, d)
    print(f"{label:22s}{duration(predicted):>20s}{duration(measured):>20s}{measured / predicted:23.2f}")
    board.append(f"{label}: predicted {duration(predicted)}, measured {duration(measured)}")

print("\nLine for the discussion board:")
print(f"Layer n = {n_layer:,}, m = {m_layer:,}, K = {K_layer:,}: {in_words(operations)} operations. " + "; ".join(board) + ".")

**How to read the result.** The last column divides the measured time by the predicted one: 1 means that the prediction was exact, 2 that the product took twice as long as predicted. The rates of Part 4 were measured on square products, with n = m = K. The time of the three loops is a prediction only, from the rate of Part 3.

**How good was the prediction?** Answer in the next cell, in a few sentences: how close did the predicted times come to the measured ones, on the CPU and on the GPU? Give one reason why a prediction and a measurement differ. Would the prediction be as good for a layer with only K = 4 neurons? Use what Part 5 showed about where the numbers are.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

<font color="#c00000" size="5"><b>Post now</b></font>, as a reply to your group's first post. One member posts for the group; begin with the names of all its members. The post contains:

- the line for the discussion board that Part 4 printed under the chart, and the chart, with one sentence that says what it shows;
- the chart of Part 5 and the line printed under it, with one sentence that says what it shows;
- the line for the discussion board that Part 6 printed.

## Then: the second notebook, and the upload

**Next, do the second notebook of meeting 12, [*Rows and columns*](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-12-rows-and-columns.ipynb).** It measures what it costs to read a matrix by rows and by columns, and needs no GPU.

**Every student uploads their own copies of both notebooks, before the meeting begins.** They are graded on the cells marked <font color="#c00000"><b>Your prompt</b></font> and <font color="#c00000"><b>Your thinking</b></font>: six in this notebook and five in the second. Check that each holds your own words and not the line that was there.

1. In each notebook, open the **File** menu, choose **Download**, and then **Download .ipynb**.
2. On Canvas, open the module of meeting 12, choose **Upload Your Class Work - Meeting 12**, and upload **both files** in the same submission.

**Before the meeting, also check the board.** Your group has two posts on the discussion board of meeting 12, the second a reply to the first, and every post names all the members of your group. Read the posts of two other groups: their machines may not be yours. The meeting starts from this work.